# PupilSense Reproduction — Left & Right Eye (Stage 1)

Runs the released ResNet50 checkpoints over the EyeDentify dataset to reproduce the left- and right-eye MAE/MAPE from *PupilSense* (Shah et al., ETRA '25).

**Before running:** Runtime > Change runtime type > pick a **GPU** (L4 or T4).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Get the package code from GitHub (public)
So the code always matches this notebook. Data + weights come from Drive.

In [ ]:
# Re-run to pull the latest code.
!rm -rf /content/EyeBiomarkers
!git clone -q https://github.com/ShirleyMgit/EyeBiomarkers.git /content/EyeBiomarkers
import sys
sys.path = [p for p in sys.path if 'code_claude/EyeBiomarkers' not in p]
sys.path.insert(0, '/content/EyeBiomarkers')
for m in [k for k in list(sys.modules) if k == 'pupilsense_repro' or k.startswith('pupilsense_repro.')]:
    del sys.modules[m]
print('package -> /content/EyeBiomarkers')

### Stage data to local disk (fast)
Reading 200k+ tiny PNGs over the Drive mount is very slow; unzip to local SSD instead. Set `RIGHT_ZIP` to your right-eye archive (leave as-is if you only have left).

In [ ]:
import shutil, zipfile, glob
from pathlib import Path

LEFT_ZIP  = '/content/drive/MyDrive/pupilsense_data_code/data/left_eyes_data.zip'   # <-- EDIT if needed
RIGHT_ZIP = '/content/drive/MyDrive/pupilsense_data_code/data/right_eyes_data.zip'  # <-- EDIT: right-eye archive

def stage(zip_path, dest):
    if not Path(zip_path).exists():
        print('skip (not found):', zip_path); return None
    dest = Path(dest)
    if dest.exists(): shutil.rmtree(dest)
    dest.mkdir(parents=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(dest)
    csvs = glob.glob(f'{dest}/**/session_data.csv', recursive=True)
    if not csvs:
        print('WARNING: no session_data.csv in', zip_path); return None
    root = Path(csvs[0]).parent.parent.parent
    print(f'{Path(zip_path).name}: {len(csvs)} sessions -> {root}')
    return root

LEFT_DATA_ROOT  = stage(LEFT_ZIP,  '/content/data_left')
RIGHT_DATA_ROOT = stage(RIGHT_ZIP, '/content/data_right')

### Run the reproduction (ResNet50) for each eye available

In [ ]:
import torch, pandas as pd
from dataclasses import replace
from pupilsense_repro.config import ReproConfig
from pupilsense_repro.runner import run_reproduction

WEIGHTS_DIR = Path('/content/drive/MyDrive/pupilsense_data_code/code/pupilsense/pre_trained_models')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

base_cfg = ReproConfig(data_root=LEFT_DATA_ROOT, weights_dir=WEIGHTS_DIR, eye='left',
                       device=DEVICE, num_workers=2,
                       results_dir=Path('/content/results'), figures_dir=Path('/content/figures'))

runs = [('left', LEFT_DATA_ROOT)]
if RIGHT_DATA_ROOT is not None:
    runs.append(('right', RIGHT_DATA_ROOT))

summaries = []
for eye, root in runs:
    cfg = replace(base_cfg, eye=eye, data_root=root)
    summaries.append(run_reproduction(cfg, bases=('resnet50',)))

summary = pd.concat(summaries, ignore_index=True)
summary

### Plots (per eye)

In [ ]:
from pupilsense_repro import plots
figs = Path('/content/figures'); figs.mkdir(parents=True, exist_ok=True)
res = Path('/content/results')
from IPython.display import Image as IPyImage, display

for eye, _ in runs:
    per_part = pd.read_csv(res / f'per_participant_mape_{eye}.csv', index_col=0)
    ppb = {b: per_part[b].dropna() for b in per_part.columns}
    plots.plot_per_participant_mape(ppb, figs / f'per_participant_mape_{eye}.png', eye=eye)
    plots.plot_mape_histogram(ppb, figs / f'mape_hist_{eye}.png')
    for base in ppb:
        preds = pd.read_csv(res / f'predictions_{eye}_{base}.csv')
        plots.plot_pred_vs_true(preds, figs / f'pred_vs_true_{eye}_{base}.png')
    display(IPyImage(str(figs / f'per_participant_mape_{eye}.png')))

## Reading the results

`summary` has one row per (eye, base) with `overall_mae`, `overall_mape`, per-participant and per-fold MAPE, and the paper's value. Per-eye CSVs land in `/content/results`.

**Caveats:** (1) the released weights are a single deployed checkpoint per eye, not per-fold models, so overall MAPE reflects train/test overlap. (2) The authors trained on **iris** crops, which are **not** in the public dataset (only eye crops), so expect ~6% here vs the paper's ~3.2% — the gap is the crop variant, not the pipeline.